In [1]:
# [셀1] import + 동결 상수
import warnings; warnings.filterwarnings("ignore")
import os, re, json, time
import numpy as np, pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

ID_COL, TARGET_COL, C20_COL = "C64", "C65", "C20"
FMT = "%Y-%m-%d %H:%M:%S.%f"
CORE10 = ["is_high_regime", "high_regime_days", "days_since_last_pm", "C33",
          "dslp_x_hour", "hour", "hour_x_c33", "C60_mean_step4", "C59_mean_step4", "is_special_recipe"]
PROTECTED = ["C17", "C11", "C31", "C15", "C16"]
SEEDS = [1, 2, 3]
SIGMA_C65 = 261.7

# 동결 참조값 (전부 로컬 venv 산, R6)
B1_LGBM_REF, B0_REF, ANCHOR = 71.366, 71.212, 70.712

# 기준선 B1_LGBM = M8_PARAMS LGBM 705 rounds
M8_PARAMS = dict(objective="regression", metric="rmse",
    learning_rate=0.029017547696366934, num_leaves=175, min_child_samples=5,
    feature_fraction=0.6324704159196377, bagging_fraction=0.864012693783303, bagging_freq=7,
    lambda_l1=5.04154328625296, lambda_l2=0.024814259264649002,
    min_split_gain=0.2573073648505903, verbose=-1, seed=42)
BASE_ROUNDS = 705

XGB_DEVICE = os.environ.get("XGB_DEVICE", "cpu")   # R6: 공식 수치는 CPU
QUICK = bool(int(os.environ.get("M6_QUICK", "0")))

def _find(name):
    for d in [".", "data", "colab_GA", "..",
              os.path.join("..", "modeling_v13"), os.path.join("..", "modeling_v13", "data"),
              os.path.join("..", "modeling_v13", "colab_GA"), os.path.join("..", "문제1(하)")]:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name} 없음 — 노트북 폴더(또는 data/·colab_GA/·../문제1(하)/)에 두세요.")

POOL = _find("v13_fdc_pool_wf_oof.csv.gz"); META = _find("core10_meta_wf.csv")
DIET = _find("feature_diet_selected.json"); SEL = _find("select_result_Conservative_GA.json")
XGBJSON = _find("tuned_params_v13_xgb.json"); TRAIN = _find("train_data.csv")
print(f"[설정] QUICK={QUICK} | XGB_DEVICE={XGB_DEVICE}")
print("xgboost", xgb.__version__, "| lightgbm", lgb.__version__)


[설정] QUICK=False | XGB_DEVICE=cpu
xgboost 3.3.0 | lightgbm 4.6.0


In [2]:
# [셀2] 헬퍼 (v2.2 stable 폴드맵 + 다중시드 lot-CV + 모델 빌더)
def _rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

def sensor_of(c):
    m = re.match(r"(C\d+)_", c)
    return m.group(1) if m else c

def floor_ok(feats):
    have = {s: sum(1 for c in feats if sensor_of(c) == s) for s in PROTECTED}
    return all(v >= 1 for v in have.values()), have

def r2_honest(rmse):
    return round(1 - (rmse / SIGMA_C65) ** 2, 4)

def stable_group_kfold(groups, n_splits=5):
    groups = np.asarray(groups)
    _, inv, cnt = np.unique(groups, return_inverse=True, return_counts=True)
    order = np.argsort(cnt, kind="stable")[::-1]
    fs = np.zeros(n_splits, dtype=np.int64); g2f = np.zeros(len(cnt), dtype=np.int64)
    for gi in order:
        f = int(np.argmin(fs)); fs[f] += cnt[gi]; g2f[gi] = f
    return g2f[inv]

def stable_splits(groups, n_splits=5):
    fv = stable_group_kfold(groups, n_splits)
    return [(np.where(fv != k)[0], np.where(fv == k)[0]) for k in range(n_splits)]

def lot_seed_splits(groups, seed, n_splits=5):
    groups = np.asarray(groups); lots = np.unique(groups); lf = np.empty(len(lots), dtype=np.int64)
    for f, (_, te) in enumerate(KFold(n_splits=n_splits, shuffle=True, random_state=seed).split(lots)):
        lf[te] = f
    l2f = {l: int(fo) for l, fo in zip(lots, lf)}
    fv = np.array([l2f[g] for g in groups])
    return [(np.where(fv != k)[0], np.where(fv == k)[0]) for k in range(n_splits)]

def make_xgb(params, rounds):
    p = dict(params)
    p.update(objective="reg:squarederror", tree_method="hist", device=XGB_DEVICE,
             random_state=42, n_estimators=int(rounds))
    return xgb.XGBRegressor(**p)

def oof_xgb(params, rounds, splits, X, y):
    oof = np.zeros(len(y))
    for tr, va in splits:
        m = make_xgb(params, rounds); m.fit(X.iloc[tr], y[tr]); oof[va] = m.predict(X.iloc[va])
    return _rmse(y, oof)

def oof_lgb(params, rounds, splits, X, y):
    oof = np.zeros(len(y))
    for tr, va in splits:
        m = lgb.train(params, lgb.Dataset(X.iloc[tr], y[tr]), num_boost_round=rounds)
        oof[va] = m.predict(X.iloc[va])
    return _rmse(y, oof)


In [3]:
# [셀3] 로드 + 확정 셋 Cons-GA(99) 재구성 + 웨이퍼 시간 + self-check
pool = pd.read_csv(POOL); pool[ID_COL] = pool[ID_COL].astype(str)
meta = pd.read_csv(META); meta[ID_COL] = meta[ID_COL].astype(str)
df = pool.merge(meta, on=ID_COL, how="inner")

# 웨이퍼 시각 = train_data C40(첫 측정) per C64  (B2 시간분할용)
tt = pd.read_csv(TRAIN, usecols=[ID_COL, "C40"]); tt[ID_COL] = tt[ID_COL].astype(str)
tt["_ts"] = pd.to_datetime(tt["C40"], format=FMT)
wf_time = tt.groupby(ID_COL)["_ts"].min().reset_index().rename(columns={"_ts": "wf_ts"})
df = df.merge(wf_time, on=ID_COL, how="left").reset_index(drop=True)

y = df[TARGET_COL].to_numpy(float); groups = df[C20_COL].to_numpy()
champions = list(json.load(open(DIET, encoding="utf-8"))["champions"]["Conservative"].values())
fixed = [f for f in dict.fromkeys(CORE10 + champions) if f in df.columns]
selp = json.load(open(SEL, encoding="utf-8"))["selected_features"]
feats = list(dict.fromkeys(fixed + selp))
X = df[feats]

ok, have = floor_ok(feats)                                              # R10
assert all(f in df.columns for f in feats), "누락 피처"
assert C20_COL not in feats and ID_COL not in feats and "fold_kf5" not in feats, "누수 피처 유입!"
assert ok and len(feats) == 99, f"셋 문제 n={len(feats)} floor={have}"
assert df["wf_ts"].notna().all(), "웨이퍼 시간 결측 — train_data C64 매칭 확인"

xgj = json.load(open(XGBJSON, encoding="utf-8"))
XGB_PARAMS = xgj["params"]; XGB_ROUNDS = int(xgj["n_estimators"])

STABLE = stable_splits(groups)
SEED_SP = {s: lot_seed_splits(groups, s) for s in SEEDS}
print(f"확정 셋 n={len(feats)} floor={have}")
print(f"df {df.shape} | unique C20 {len(np.unique(groups))} lot | 기간 {str(df['wf_ts'].min())[:10]} ~ {str(df['wf_ts'].max())[:10]}")
print(f"챔피언 XGB: rounds={XGB_ROUNDS} | params={json.dumps(XGB_PARAMS)}")

if not QUICK:
    _b = oof_lgb(M8_PARAMS, BASE_ROUNDS, STABLE, X, y)
    _flag = "OK" if abs(_b - B1_LGBM_REF) < 0.05 else "주의 — env/핀 점검"
    print(f"[self-check] baseline LGBM stable GKF = {_b:.3f} (동결 {B1_LGBM_REF}, Δ{_b-B1_LGBM_REF:+.3f}) {_flag}")


확정 셋 n=99 floor={'C17': 4, 'C11': 5, 'C31': 3, 'C15': 3, 'C16': 2}
df (11939, 668) | unique C20 1217 lot | 기간 2018-12-01 ~ 2019-02-08
챔피언 XGB: rounds=246 | params={"learning_rate": 0.027015944979119196, "max_depth": 4, "min_child_weight": 6.891957479431703, "subsample": 0.8181242434283986, "colsample_bytree": 0.8782954149928492, "reg_alpha": 0.07630553997211231, "reg_lambda": 0.05392394823978756, "gamma": 2.353287110798049}
[self-check] baseline LGBM stable GKF = 71.366 (동결 71.366, Δ+0.000) OK


In [4]:
# [셀4] B1 — 다중시드 Lot-CV: 챔피언 XGB vs 기준선 LGBM(B1_LGBM)  ①′ 보조축
print("=== B1 다중시드 Lot-CV (챔피언 XGBoost vs 기준선 B1_LGBM) ===")
seeds_run = SEEDS[:1] if QUICK else SEEDS
b1 = []
for s in seeds_run:
    sp = SEED_SP[s]
    champ = oof_xgb(XGB_PARAMS, XGB_ROUNDS, sp, X, y)
    base = oof_lgb(M8_PARAMS, BASE_ROUNDS, sp, X, y)
    b1.append(dict(seed=s, champ=round(champ, 3), base=round(base, 3), improve=round(base - champ, 3)))
    print(f"  seed{s}: 챔피언 {champ:.3f}  vs  기준선 {base:.3f}  →  개선 {base-champ:+.3f}")
imp = [r["improve"] for r in b1]
mean_imp = float(np.mean(imp)); worst_imp = float(np.min(imp))
b1_pass = (mean_imp >= 0.5) and (worst_imp >= 0)
champ_mean = float(np.mean([r["champ"] for r in b1]))
print(f"  → 평균 개선 {mean_imp:+.3f} | 최악 {worst_imp:+.3f} | 챔피언 시드평균 {champ_mean:.3f} (R²={r2_honest(champ_mean)})")
print(f"  → B1 게이트(평균≥0.5 ∧ 최악≥0): {'PASS' if b1_pass else 'FAIL'}")


=== B1 다중시드 Lot-CV (챔피언 XGBoost vs 기준선 B1_LGBM) ===
  seed1: 챔피언 67.661  vs  기준선 73.232  →  개선 +5.572
  seed2: 챔피언 66.547  vs  기준선 69.870  →  개선 +3.323
  seed3: 챔피언 66.728  vs  기준선 70.905  →  개선 +4.177
  → 평균 개선 +4.357 | 최악 +3.323 | 챔피언 시드평균 66.979 (R²=0.9345)
  → B1 게이트(평균≥0.5 ∧ 최악≥0): PASS


In [5]:
# [셀5] B2 — 시간분할 (미래 Lot 외삽). ⚠️ 홀드아웃 조회 1/2회차 소비 (R5)
print("=== B2 시간분할 (조회 1/2회차) — 챔피언99 vs core10 단독 ===")
lot_ts = df.groupby(C20_COL)["wf_ts"].median().sort_values()
lots_sorted = lot_ts.index.to_numpy()
core_feats = [f for f in CORE10 if f in df.columns]
PM1 = pd.Timestamp("2018-12-24")   # 첫 major PM (pm_log)
b2 = []
for frac in [0.6, 0.7, 0.8]:
    k = int(len(lots_sorted) * frac)
    tr_lots = set(lots_sorted[:k].tolist())
    trm = df[C20_COL].isin(tr_lots).to_numpy()
    tri = np.where(trm)[0]; tei = np.where(~trm)[0]
    cut_ts = lot_ts.iloc[k - 1]
    ch = make_xgb(XGB_PARAMS, XGB_ROUNDS); ch.fit(X.iloc[tri], y[tri]); r_ch = _rmse(y[tei], ch.predict(X.iloc[tei]))
    co = make_xgb(XGB_PARAMS, XGB_ROUNDS); co.fit(df[core_feats].iloc[tri], y[tri]); r_co = _rmse(y[tei], co.predict(df[core_feats].iloc[tei]))
    crosses_pm = bool(cut_ts < PM1 <= df["wf_ts"].max())
    b2.append(dict(cut=f"{int(frac*100)}/{int((1-frac)*100)}", cut_date=str(cut_ts)[:10],
                   n_test=int(tei.size), champ99=round(r_ch, 3), core10=round(r_co, 3), win=bool(r_ch <= r_co)))
    print(f"  {int(frac*100)}/{int((1-frac)*100)} (컷 {str(cut_ts)[:10]}, test {int(tei.size)}): 챔피언99 {r_ch:.3f}  vs  core10 {r_co:.3f}  → {'우위' if r_ch<=r_co else '열세'}")
b2_pass = all(r["win"] for r in b2)
worst_b2 = max(r["champ99"] for r in b2)
print(f"  → 전 컷 우위: {'PASS' if b2_pass else 'FAIL'} | 시간분할 최악 champ99 {worst_b2:.3f} (참고 lot-CV≈66.98; 파국 열화 여부는 강건 판정)")


=== B2 시간분할 (조회 1/2회차) — 챔피언99 vs core10 단독 ===
  60/40 (컷 2019-01-14, test 4966): 챔피언99 104.933  vs  core10 106.043  → 우위
  70/30 (컷 2019-01-21, test 3786): 챔피언99 97.956  vs  core10 92.741  → 열세
  80/19 (컷 2019-01-26, test 2655): 챔피언99 76.543  vs  core10 76.951  → 우위
  → 전 컷 우위: FAIL | 시간분할 최악 champ99 104.933 (참고 lot-CV≈66.98; 파국 열화 여부는 강건 판정)


In [6]:
# [셀6] B3 — 레짐 분해 (챔피언 stable GKF-OOF 오차 층화). 리포트용.
print("=== B3 레짐 분해 (챔피언 stable GKF-OOF) ===")
oof = np.zeros(len(y))
for tr, va in STABLE:
    m = make_xgb(XGB_PARAMS, XGB_ROUNDS); m.fit(X.iloc[tr], y[tr]); oof[va] = m.predict(X.iloc[va])
b3_oof_rmse = round(_rmse(y, oof), 3)
print(f"  전체 stable GKF RMSE = {b3_oof_rmse} (R²={r2_honest(b3_oof_rmse)})")

def _stratum(mask, label):
    n = int(mask.sum())
    if n > 0:
        print(f"    {label:<26} n={n:>5}  RMSE={_rmse(y[mask], oof[mask]):.3f}")

if "is_high_regime" in df.columns:
    for v in [0, 1]:
        _stratum(df["is_high_regime"].to_numpy() == v, f"is_high_regime={v}")
dslp = df["days_since_last_pm"].to_numpy()
qs = np.quantile(dslp, [.25, .5, .75])
bounds = [(-np.inf, qs[0]), (qs[0], qs[1]), (qs[1], qs[2]), (qs[2], np.inf)]
for i, (lo, hi) in enumerate(bounds):
    _stratum((dslp > lo) & (dslp <= hi), f"days_since_pm Q{i+1}")


=== B3 레짐 분해 (챔피언 stable GKF-OOF) ===
  전체 stable GKF RMSE = 66.743 (R²=0.935)
    is_high_regime=0           n= 3692  RMSE=36.015
    is_high_regime=1           n= 8247  RMSE=76.604
    days_since_pm Q1           n= 2985  RMSE=67.784
    days_since_pm Q2           n= 2985  RMSE=65.055
    days_since_pm Q3           n= 2984  RMSE=64.725
    days_since_pm Q4           n= 2985  RMSE=69.297


In [7]:
# [셀7] B4 — 물리 근거 (챔피언 XGB gain importance). 리포트용.
print("=== B4 물리 근거 (XGB gain importance, 전체 적합) ===")
mfull = make_xgb(XGB_PARAMS, XGB_ROUNDS); mfull.fit(X, y)
score = mfull.get_booster().get_score(importance_type="gain")
# DataFrame 적합 시 키 = 피처명. (혹시 f0.. 형태면 인덱스 복원)
if score and all(k.startswith("f") and k[1:].isdigit() for k in score):
    score = {feats[int(k[1:])]: v for k, v in score.items()}
imp = pd.Series(score).reindex(feats).fillna(0.0).sort_values(ascending=False)
tot = float(imp.sum()) or 1.0
print("  상위 15 (gain 비중):")
for c in imp.index[:15]:
    print(f"    {c:<24} {imp[c]/tot*100:5.1f}%")
prot_share = float(sum(imp.get(c, 0) for c in imp.index if sensor_of(c) in PROTECTED)) / tot * 100
b4_prot_share = round(prot_share, 1)
print(f"  필수 5센서(C17·C11·C31·C15·C16) 중요도 비중 = {b4_prot_share}%")
print("  ⚠️ 상위 기여 물리명 미확정 후보: C25(노후드리프트)·C59/C60(멀티플렉스)·C63 → 데이터 사전 v3 멘토 우선순위(① C31 단위 ② C63 정체 ③ C59/C60 물리량)")


=== B4 물리 근거 (XGB gain importance, 전체 적합) ===
  상위 15 (gain 비중):
    is_high_regime            76.9%
    C11_min_step4             10.7%
    days_since_last_pm         2.7%
    high_regime_days           2.6%
    C33                        1.6%
    is_special_recipe          1.2%
    C31_mean_step4             1.1%
    C17_max_step4              0.6%
    hour_x_c33                 0.1%
    hour                       0.1%
    dslp_x_hour                0.1%
    C62_mean_step1             0.1%
    C61_last_step7             0.1%
    C61_last_step1             0.1%
    C12_mean_step1             0.1%
  필수 5센서(C17·C11·C31·C15·C16) 중요도 비중 = 12.7%
  ⚠️ 상위 기여 물리명 미확정 후보: C25(노후드리프트)·C59/C60(멀티플렉스)·C63 → 데이터 사전 v3 멘토 우선순위(① C31 단위 ② C63 정체 ③ C59/C60 물리량)


In [8]:
# [셀8] B5 운영 전제 + G6′ 종합(참고) + 저장
def _has(n):
    try:
        _find(n); return True
    except Exception:
        return False

print("=== B5 운영 전제 (체크리스트) ===")
b5 = {"pm_log 연동": _has("pm_log.json"),
      "결측 step 거동": "XGB native NaN 처리 (대치 불요)",
      "입력 스키마": f"{len(feats)}피처 Cons-GA(99) 고정 · floor={have}",
      "재현 패키지": "tuned_params_v13_xgb.json + build_fdc_pool.py + 본 노트북"}
for k, v in b5.items():
    print(f"    {k}: {v}")

print("\n=== G6′ 종합 (참고 — 공식은 강건이 회신 숫자로) ===")
r2_ok = r2_honest(b3_oof_rmse) >= 0.9
g = {"B1 (평균≥0.5 ∧ 최악≥0)": b1_pass,
     "B2 (전 컷 core10 우위)": b2_pass,
     "R² ≥ 0.9": r2_ok,
     "B3~B5 리포트 완비": True}
for k, v in g.items():
    print(f"    {k}: {'✅' if v else '❌'}")
print(f"    → G6′ 잠정: {'PASS 권고' if all(g.values()) else 'MIXED — 강건 판정 필요'}")

out = dict(champion="XGBoost", set="Conservative-GA(99)", n_feats=len(feats),
           quick=QUICK,
           B1=dict(rows=b1, mean_improve=round(mean_imp, 3), worst_improve=round(worst_imp, 3), pass_=b1_pass),
           B2=dict(rows=b2, pass_=b2_pass, worst_champ99=worst_b2, note="홀드아웃 조회 1/2회차 소비(R5)"),
           B3=dict(stable_oof_rmse=b3_oof_rmse, R2=r2_honest(b3_oof_rmse)),
           B4=dict(protected_share_pct=b4_prot_share),
           refs=dict(B1_LGBM=B1_LGBM_REF, B0=B0_REF, anchor=ANCHOR),
           note="공식 G6′ 판정은 강건이 회신 숫자로. 이 파일은 검산·산출 인덱스.")
json.dump(out, open("field_battery_results.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("\nsaved: field_battery_results.json")


=== B5 운영 전제 (체크리스트) ===
    pm_log 연동: True
    결측 step 거동: XGB native NaN 처리 (대치 불요)
    입력 스키마: 99피처 Cons-GA(99) 고정 · floor={'C17': 4, 'C11': 5, 'C31': 3, 'C15': 3, 'C16': 2}
    재현 패키지: tuned_params_v13_xgb.json + build_fdc_pool.py + 본 노트북

=== G6′ 종합 (참고 — 공식은 강건이 회신 숫자로) ===
    B1 (평균≥0.5 ∧ 최악≥0): ✅
    B2 (전 컷 core10 우위): ❌
    R² ≥ 0.9: ✅
    B3~B5 리포트 완비: ✅
    → G6′ 잠정: MIXED — 강건 판정 필요

saved: field_battery_results.json
